# IST Scenario 3 — Which admission characteristics predict a poor outcome?

_From a randomised trial to observational, prognostic questions. Level: third analysis. About two hours._

**Learning goals.** Select a cohort with filters and justify the selection; compare a continuous
variable between two groups with a t-test; test several categorical factors one by one; build a
multivariable logistic regression and read adjusted odds ratios; explain confounding and why
adjustment matters here but hardly mattered in Scenario 1; judge a published prognostic score.

**Clinical background.** Randomisation answers one question, the effect of the allocated
treatment. But the same dataset records what every patient looked like at admission: age, sex,
consciousness, pattern of deficits, atrial fibrillation, blood pressure. These were measured
before any treatment, so they can answer an observational question: who does badly? The IST
investigators built such a prognostic model and stored its prediction in the dataset
(`Predicted death or dependence at 6 months`).

**Today.** The predictors you will find are still the main prognostic factors in stroke, but
the absolute risks are 1990s risks: a drowsy 80-year-old with a total anterior circulation
syndrome had almost no chance of independence in IST; with thrombectomy, some such patients now
recover. Severity is now measured with the NIHSS and outcome with the modified Rankin Scale.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. Cohort and variables

**Cohort:** patients with an ischaemic stroke confirmed by CT before randomisation, filter
`CT-confirmed ischaemic stroke = 1` (curation-derived). This is the restriction the IST authors
recommend for analyses of definite ischaemic stroke. Expected 12,550 patients, 12,447 with a
known six-month outcome. Why filter here and not in Scenarios 1 and 2? A treatment effect must
be analysed in everyone randomised; a prognostic question about ischaemic stroke should be asked
in patients who really had one.

| Role | Variable in the tree | Codes |
|---|---|---|
| cohort filter | CT-confirmed ischaemic stroke (`CTISCH`) | 1 |
| outcome | Dead or dependent at 6 months (`DEADDEP6`) | 0, 1 |
| predictors | Age; Age group (`AGEGRP`: 1 <60, 2 60-69, 3 70-79, 4 80+); Sex (1 male, 2 female); Conscious level (1 alert, 2 drowsy, 3 unconscious); OCSP stroke syndrome (`STYPE`: 1 TACS, 2 PACS, 3 POCS, 4 LACS, 5 other); Atrial fibrillation (0, 1; empty in the pilot phase); Systolic blood pressure; Number of deficits (`NDEF`, 0 to 8); Infarct visible on CT (0, 1) | |
| the IST model | Predicted death or dependence at 6 months (`EXPDD`) | probability 0 to 1 |

In [ ]:
ctisch  = dm.variables["CT-confirmed ischaemic stroke"]
dd6     = dm.variables["Dead or dependent at 6 months"]
age     = dm.variables["Age"]
agegrp  = dm.variables["Age group"]
sex     = dm.variables["Sex"]
rconsc  = dm.variables["Conscious level"]
stype   = dm.variables["OCSP stroke syndrome"]
ratrial = dm.variables["Atrial fibrillation"]
rsbp    = dm.variables["Systolic blood pressure"]
ndef    = dm.variables["Number of deficits"]
rvisinf = dm.variables["Infarct visible on CT"]
expdd   = dm.variables["Predicted death or dependence at 6 months"]

cohort = F(ctisch) == "1"                    # kept in every step of this scenario
display(mip.to_frame([ctisch, dd6, age, agegrp, sex, rconsc, stype, ratrial, rsbp, ndef, rvisinf, expdd]))

## Step 1. Describe the cohort

*Expected:* 12,550 patients; 60.1% dead or dependent at six months (12,447 with known status), a
little lower than in the whole trial because haemorrhages and unscanned comatose patients are
excluded.

In [ ]:
base = analysis([age, sex, rconsc, stype, ratrial, dd6], cohort).describe()
display(numeric_summary(base, [age]).style.format({"mean": "{:.1f}", "sd": "{:.1f}"}).hide(axis="index"))
for v in (sex, rconsc, stype, ratrial, dd6):
    display(category_counts(base, v).style.format({"percent": "{:.1f}"}).hide(axis="index"))

## Step 2. Age as a continuous variable: Independent t-test

Mean age in patients dead or dependent versus independent survivors. The test returns the
t statistic, the p-value, the mean difference with its confidence interval and Cohen's d; the
two group means come from a `describe()` per group.

*Expected:* 73.5 versus 66.3 years, a difference of 7.2 years (95% CI 6.7 to 7.6); t about 34;
p far below 0.001; Cohen's d 0.63, a medium to large effect.

In [ ]:
ttest = analysis([age, dd6], cohort & F(dd6).is_not_null(), drop_missing=True).t_test(
    variable=age, group_by=dd6, group_a="1", group_b="0")
display(ttest.to_frame().style.format("{:.3f}"))

means, _ = describe_by([age], group=dd6, base_filter=cohort)
display(means.style.format("{:.1f}"))

## Step 3. Age as groups

*Expected:* dead or dependent in 39.0% under 60, 51.0% at 60-69, 62.9% at 70-79, 79.5% at 80 and
over. Each decade adds 11 to 17 percentage points of risk.

In [ ]:
show_crosstab(crosstab(agegrp, dd6, base_filter=cohort), "Dead or dependent at 6 months, by age group")

## Step 4. One factor at a time

*Expected percentages dead or dependent:* conscious level alert 50.3%, drowsy 87.4%, unconscious
98.8%; OCSP syndrome TACS 85.7%, PACS 58.5%, POCS 47.2%, LACS 43.0%; atrial fibrillation no 56.6%,
yes 78.6%; sex men 53.8%, women 67.5%; infarct visible on CT no 53.5%, yes 66.8%. All p < 0.001.

Every factor is "significant": with 12,000 patients almost anything is. Look at the sizes:
consciousness and syndrome separate patients by 40 to 50 percentage points, sex by 14.

In [ ]:
factors = [rconsc, stype, ratrial, sex, rvisinf]
one_at_a_time = []
for v in factors:
    t = crosstab(v, dd6, base_filter=cohort)
    for _, row in t.iterrows():
        one_at_a_time.append({"factor": v.label, "level": row[v.label], "n": row["n"],
                              "% dead or dependent": row["% Yes"], "p (chi-square)": t.attrs["p"]})
one_at_a_time = pd.DataFrame(one_at_a_time)
display(one_at_a_time.style.format({"% dead or dependent": "{:.1f}", "p (chi-square)": fmt_p}).hide(axis="index"))

fig, axes = plt.subplots(1, len(factors), figsize=(16, 3.5), sharey=True)
for ax, v in zip(axes, factors):
    sub = one_at_a_time[one_at_a_time["factor"] == v.label]
    ax.bar([str(l).split(" - ")[0][:12] for l in sub["level"]], sub["% dead or dependent"], color="#4C72B0")
    ax.set_title(v.label, fontsize=10)
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
axes[0].set_ylabel("% dead or dependent")
plt.tight_layout()
plt.show()

## Step 5. Why one factor at a time is not enough

Women in IST were older than men, and patients with atrial fibrillation were older and more
often had a TACS. Part of the "effect" of sex or of atrial fibrillation in Step 4 may be age or
severity in disguise: this is **confounding**.

*Expected:* women about 73 years on average, men about 68. The five-year gap explains part of
the sex difference in outcome.

In [ ]:
num, cat = describe_by([age, stype], group=sex, base_filter=cohort)
display(num.style.format("{:.1f}"))
num, cat = describe_by([age, stype], group=ratrial, base_filter=cohort)
display(num.style.format("{:.1f}"))
display(cat.style.format("{:.1f}"))

## Step 6. Multivariable logistic regression

Dependent variable `Dead or dependent at 6 months` (event = 1); covariates age, sex, conscious
level, OCSP syndrome, atrial fibrillation, systolic blood pressure, number of deficits. The
platform uses the first level of each categorical covariate as the **reference**: male, fully
alert, TACS, no atrial fibrillation. The 984 pilot-phase patients without atrial fibrillation
data are dropped automatically (about 11,950 patients remain).

*Expected adjusted odds ratios:* age 1.05 per year; female 1.34; drowsy 4.1 and unconscious
about 70 versus alert (very wide interval, few patients); PACS 0.60, POCS 0.40 and LACS 0.56
versus TACS (the protocol quotes the same contrasts with LACS as reference: TACS 1.78, PACS 1.07,
POCS 0.71); atrial fibrillation 1.43; systolic pressure 0.998 per mmHg; number of deficits 1.46
per deficit.

Read every odds ratio as "holding the other variables constant". The sex effect shrank from an
unadjusted 1.8 to 1.34: part of it was age. Atrial fibrillation shrank from 2.8 to 1.43: much of
it was age and severity. Blood pressure has practically no independent effect.

In [ ]:
covariates = [age, sex, rconsc, stype, ratrial, rsbp, ndef]
model = analysis([dd6] + covariates, cohort & F(dd6).is_not_null(), drop_missing=True).logistic_regression(
    x=covariates, y=dd6, positive_class="1")

display(model)
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Adjusted odds of death or dependency at 6 months (CT-confirmed ischaemic stroke)")

# model fit, if the platform returns it
fit = model.raw.get("summary", {}) if isinstance(model.raw, dict) else {}
print({k: fit.get(k) for k in ("n_obs", "ll", "ll0", "aic", "bic", "r_squared_mcf") if k in fit})

## Step 7. How good is the published IST prognostic model?

The IST team stored each patient's predicted probability of death or dependency in
`Predicted death or dependence at 6 months`. Compare its mean between patients who were dead or
dependent and those who were independent.

*Expected:* mean predicted probability 0.72 versus 0.48. The prediction separates the two groups
clearly but far from perfectly: many independent survivors had a predicted risk above 0.5. This
is typical of prognostic models in stroke: good for groups, uncertain for the individual.

In [ ]:
pred, _ = describe_by([expdd], group=dd6, base_filter=cohort)
display(pred.style.format("{:.2f}"))

ttest_pred = analysis([expdd, dd6], cohort & F(dd6).is_not_null(), drop_missing=True).t_test(
    variable=expdd, group_by=dd6, group_a="1", group_b="0")
display(ttest_pred.to_frame().style.format("{:.3f}"))

## Step 8. State the result

> Among 12,447 IST patients with CT-confirmed ischaemic stroke, 60% were dead or dependent at
> six months. Independent predictors of a poor outcome were older age (odds ratio 1.05 per year),
> reduced consciousness (4-fold if drowsy, far higher if unconscious), a total anterior
> circulation syndrome (the other syndromes had 0.4 to 0.6 times its odds), a larger number of
> deficits (1.5 per deficit), atrial fibrillation (1.4) and female sex (1.3). Blood pressure at
> admission was not independently predictive. These factors remain the core of stroke prognosis
> today; the absolute risks reflect care before reperfusion therapy and stroke units.

**Questions for discussion**

1. In Scenario 1 adjustment barely changed the aspirin odds ratio; here it halved the atrial
   fibrillation odds ratio. Explain the difference in one sentence using the word "randomisation".
2. Female sex remains a predictor after adjustment. List three explanations, from biological to
   methodological (think about what "dependent" means and who lives alone).
3. `Number of deficits` treats "cannot assess" as absent. Which patients are most often "cannot
   assess"? In which direction could this bias its coefficient?
4. Would you use this model to tell a family what to expect? Why not?

## Optional extensions

- **Outcome as an ordered scale.** `Six-month outcome` has four ordered levels: dead, dependent,
  not recovered, fully recovered. Cross it against conscious level to see the whole distribution
  shift, not just the dichotomy.
- **The weekend effect.** Cross `Weekend randomisation` against the outcome in the whole trial
  (no cohort filter). Expected: 60.9% on weekends versus 63.4% on weekdays. Before concluding that
  weekends are safer, look at age and consciousness by weekend, and think about which patients
  get randomised into a trial on a Sunday.

In [ ]:
occode = dm.variables["Six-month outcome"]
rweekend = dm.variables["Weekend randomisation"]

full = crosstab(rconsc, occode, base_filter=cohort, event="1")
display(full.style.format({"% Dead": "{:.1f}"}).hide(axis="index"))
dist = full.set_index("Conscious level").drop(columns=["n", "% Dead"])
ax = dist.div(dist.sum(axis=1), axis=0).mul(100).plot(kind="barh", stacked=True, figsize=(8, 3),
                                                       color=["#C44E52", "#DD8452", "#4C72B0", "#55A868"])
ax.set_xlabel("% of patients")
ax.set_ylabel("")
ax.set_title("Six-month outcome by conscious level at randomisation")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

show_crosstab(crosstab(rweekend, dd6), "Dead or dependent at 6 months, weekend vs weekday randomisation")
num, cat = describe_by([age, rconsc], group=rweekend)
display(num.style.format("{:.1f}"))
display(cat.style.format("{:.1f}"))